# Fabric CLI — List & Create Workspaces

This notebook runs the same commands as `workspace-commands.txt`, but from a **Fabric notebook** using the built-in `!fab` shell magic.

Starting with **Fabric CLI v1.5 (GA)** the CLI is available in Fabric notebooks **without any `pip install`** — just prefix any command with `!fab`. Authentication uses the notebook's running identity (user or service principal), so no `fab auth login` is required.

Reference: [Fabric CLI v1.5 is here — Generally Available](https://community.fabric.microsoft.com/t5/Fabric-Updates-Blogs/Fabric-CLI-v1-5-is-here-Generally-Available/ba-p/5172135)

Docs: https://microsoft.github.io/fabric-cli/

> Each `!fab` cell runs as a one-shot subprocess, so the interactive shell's working directory does not persist between cells. Use absolute paths (e.g. `/WS - Sales.Workspace`) instead of relying on `cd`.

## Configuration

Set the capacity and (optional) security group used by the commands below.

In [ ]:
CAPACITY_NAME = "<your-capacity-name>"
GROUP_ID      = "<your-group-id>"   # Leave as-is to skip the ACL step

# Verify the CLI is available (built into the Fabric notebook runtime since v1.5 GA)
!fab --version

## 1. List workspaces

List all workspaces visible to the running identity.

In [ ]:
!fab ls

## 2. Create workspaces

Create three workspaces and assign them to the configured capacity by name.

In [ ]:
!fab mkdir "WS - Sales.Workspace"     -P capacityName=$CAPACITY_NAME
!fab mkdir "WS - Marketing.Workspace" -P capacityName=$CAPACITY_NAME
!fab mkdir "WS - Finance.Workspace"   -P capacityName=$CAPACITY_NAME

## 3. Assign a security group as Workspace Member

Skip this cell if no group assignment is needed.

In [ ]:
!fab acl set "WS - Sales.Workspace"     -I $GROUP_ID -R member --force
!fab acl set "WS - Marketing.Workspace" -I $GROUP_ID -R member --force
!fab acl set "WS - Finance.Workspace"   -I $GROUP_ID -R member --force

## 4. Create an Internet Sales workspace, import a notebook, then copy it across

Creates a fourth workspace, imports the `Rebrickable - Ingest` notebook into the **Sales** workspace, and copies it from there into **Internet Sales**.

In [ ]:
!fab mkdir "WS - Internet Sales.Workspace" -P capacityName=$CAPACITY_NAME

# Import the notebook into the Sales workspace
!fab import "/WS - Sales.Workspace/Rebrickable - Ingest.Notebook" -i "Demos/Resources/Rebrickable - Ingest.Notebook/Rebrickable - Ingest.ipynb" --format .ipynb --force

# Copy the notebook from Sales to Internet Sales
!fab cp "/WS - Sales.Workspace/Rebrickable - Ingest.Notebook" "/WS - Internet Sales.Workspace/Rebrickable - Ingest.Notebook"

## 5. Get the Internet Sales workspace ID and list its items via the REST API

Reads the workspace ID with `get -q` and passes it to the `api` command, which calls the [Items - List Items](https://learn.microsoft.com/en-us/rest/api/fabric/core/items/list-items) REST endpoint.

In [ ]:
# Capture the workspace ID into a Python variable, then call the REST API
workspace_id = !fab get "WS - Internet Sales.Workspace" -q id
workspace_id = workspace_id[-1].strip()
print(f"Internet Sales workspace ID: {workspace_id}")

!fab api "/workspaces/$workspace_id/items"

## 6. Export the notebook from Internet Sales to a local folder

In [ ]:
!fab export "/WS - Internet Sales.Workspace/Rebrickable - Ingest.Notebook" -o "Demos/Resources/Exports" --format .ipynb --force

## 7. Clean up — delete the workspaces

The interactive `rm .` form from the shell relies on a selection prompt and won't work in a non-interactive notebook cell. Delete each workspace explicitly with `--force` instead.

In [ ]:
!fab rm "/WS - Sales.Workspace"          --force
!fab rm "/WS - Marketing.Workspace"      --force
!fab rm "/WS - Finance.Workspace"        --force
!fab rm "/WS - Internet Sales.Workspace" --force